# t-SNE and UMAP: Non-Linear Dimensionality Reduction

t-SNE (t-Distributed Stochastic Neighbor Embedding) and UMAP (Uniform Manifold Approximation and Projection) are non-linear dimensionality reduction methods for visualization. They emphasize preservation of local neighborhoods in contrast to PCA, which preserves global variance.

**Comparison (brief):**

| Aspect | PCA | t-SNE | UMAP |
|--------|-----|-------|------|
| Type | Linear | Non-linear | Non-linear |
| Structure | Global | Local | Local + Global |
| Speed | Fast | Slow | Medium |
| Scalability | High | Low | Moderate |
| Use | Preprocessing, compression | Visualization | Visualization & exploration

---

## t-SNE (summary)

t-SNE minimizes the KL divergence between similarity distributions in the original and embedding spaces. High-dimensional similarities are modeled with a Gaussian kernel, low-dimensional similarities with a Student t-distribution. Optimization is performed by gradient descent.

## UMAP (summary)

UMAP constructs a fuzzy topological representation of the data and optimizes a low-dimensional representation by minimizing a cross-entropy between fuzzy simplicial sets. UMAP is typically faster than t-SNE and preserves more of the global structure.

## Key parameters

- `perplexity` (t-SNE): effective neighborhood size (typically 5–50).
- `n_neighbors` (UMAP): neighborhood size controlling local structure (default 15).
- `n_iter`: optimization iterations (typical 1000–5000).
- `learning_rate`: gradient descent step size.
- `random_state`: seed for reproducibility.

---

## Implementation in Python

In [13]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris, make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style("darkgrid")
plt.rcParams['figure.figsize'] = (14, 6)

# Create a dataset with clear clusters (3D data)
# This simulates a more realistic high-dimensional dataset
np.random.seed(42)
X, y = make_blobs(n_samples=300, centers=4, n_features=10, 
                   cluster_std=1.5, random_state=42)

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Dataset shape:", X_scaled.shape)
print("Number of classes:", len(np.unique(y)))
print("Class distribution:", np.bincount(y))

Dataset shape: (300, 10)
Number of classes: 4
Class distribution: [75 75 75 75]


In [14]:
# Apply PCA for comparison
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print("\nPCA Results:")
print(f"Variance explained: {pca.explained_variance_ratio_.sum():.1%}")
print(f"PC1: {pca.explained_variance_ratio_[0]:.1%}, PC2: {pca.explained_variance_ratio_[1]:.1%}")


PCA Results:
Variance explained: 79.6%
PC1: 59.1%, PC2: 20.5%


In [15]:
# Apply t-SNE with different perplexity values and UMAP
print("\nApplying dimensionality reduction methods...")

# t-SNE with perplexity 30
tsne_30 = TSNE(n_components=2, perplexity=30, n_iter=1000, 
               random_state=42, learning_rate=200, verbose=0)
X_tsne_30 = tsne_30.fit_transform(X_scaled)

# t-SNE with higher perplexity (more global)
tsne_50 = TSNE(n_components=2, perplexity=50, n_iter=1000,
               random_state=42, learning_rate=200, verbose=0)
X_tsne_50 = tsne_50.fit_transform(X_scaled)

# t-SNE with low perplexity (more local)
tsne_5 = TSNE(n_components=2, perplexity=5, n_iter=1000,
              random_state=42, learning_rate=200, verbose=0)
X_tsne_5 = tsne_5.fit_transform(X_scaled)

# UMAP (install if missing)
try:
    import umap
except Exception:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'umap-learn', '-q'])
    import umap

reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
X_umap = reducer.fit_transform(X_scaled)

print("Methods applied successfully.")


Applying dimensionality reduction methods...



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Methods applied successfully.


In [17]:
# Visualization: Original Data and Dimensionality Reduction Methods
fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)

# Original data (10D projected to 2D using PCA for visualization)
ax1 = fig.add_subplot(gs[0, 0])
ax1.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', s=80, edgecolor='k', alpha=0.7)
ax1.set_xlabel('Dimension 1 (PCA projection for visualization)')
ax1.set_ylabel('Dimension 2 (PCA projection for visualization)')
ax1.set_title('Original 10D Data\nProjected with PCA', fontweight='bold')
ax1.grid(True, alpha=0.3)

# PCA (true reduction)
ax2 = fig.add_subplot(gs[0, 1])
ax2.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', s=80, edgecolor='k', alpha=0.7)
ax2.set_xlabel('PC1')
ax2.set_ylabel('PC2')
ax2.set_title('PCA (Linear)', fontweight='bold')
ax2.grid(True, alpha=0.3)

# t-SNE with perplexity=30
ax3 = fig.add_subplot(gs[0, 2])
ax3.scatter(X_tsne_30[:, 0], X_tsne_30[:, 1], c=y, cmap='viridis', s=80, edgecolor='k', alpha=0.7)
ax3.set_xlabel('t-SNE Dimension 1')
ax3.set_ylabel('t-SNE Dimension 2')
ax3.set_title('t-SNE (perplexity=30)', fontweight='bold')
ax3.grid(True, alpha=0.3)

# t-SNE with perplexity=50
ax4 = fig.add_subplot(gs[1, 0])
ax4.scatter(X_tsne_50[:, 0], X_tsne_50[:, 1], c=y, cmap='viridis', s=80, edgecolor='k', alpha=0.7)
ax4.set_xlabel('t-SNE Dimension 1')
ax4.set_ylabel('t-SNE Dimension 2')

ax4.set_title('t-SNE (perplexity=50)', fontweight='bold')plt.show()

ax4.grid(True, alpha=0.3)

         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# UMAP         fontsize=10, verticalalignment='top', family='monospace',

if umap_available:ax6.text(0.05, 0.95, comparison_text, transform=ax6.transAxes,

    ax5 = fig.add_subplot(gs[1, 1])- Better scalability"""

    ax5.scatter(X_umap[:, 0], X_umap[:, 1], c=y, cmap='viridis', s=80, edgecolor='k', alpha=0.7)- Preserves local + global

    ax5.set_xlabel('UMAP Dimension 1')- Non-linear, medium speed

    ax5.set_ylabel('UMAP Dimension 2')UMAP:

    ax5.set_title('UMAP (n_neighbors=15)', fontweight='bold')

    ax5.grid(True, alpha=0.3)- Excellent for visualization

- Preserves local structure

# Add a comparison text box- Non-linear, slow

ax6 = fig.add_subplot(gs[1, 2])t-SNE:

ax6.axis('off')

comparison_text = """METHOD COMPARISON- ~80% variance explained

- Preserves global structure

PCA:- Linear, fast

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 45)

---

## Plot Analysis

Original Data: 10-dimensional data projected with PCA for 2D visualization. Clusters are present but boundaries are diffuse.

PCA: Linear transformation preserves global structure. Clusters overlap due to variance constraint.

t-SNE (perplexity=30): Non-linear transformation creates clear cluster separation by emphasizing local neighborhoods. Best balance for visualization.

t-SNE (perplexity=50): Higher perplexity considers broader neighborhoods, resulting in slightly different cluster arrangement but similar separability.

UMAP: Combines local and global structure preservation. Faster than t-SNE with competitive visualization quality and better scalability.

## ✅ Advantages of t-SNE

| Advantage | Benefit |
|-----------|---------|
| **Excellent Visualization** | Creates beautiful, interpretable 2D/3D plots |
| **Preserves Local Structure** | Similar points stay close (discovers neighborhoods) |
| **Non-Linear** | Captures complex, curved relationships PCA misses |
| **Cluster Separation** | Different classes visibly separated in output |
| **Discovers Hidden Patterns** | Finds meaningful groupings in unlabeled data |

## ❌ Disadvantages of t-SNE

| Limitation | Impact |
|-----------|--------|
| **Computationally Expensive** | O(N²) complexity → slow on 100K+ points |
| **Non-Deterministic** | Different runs = different results (fix random_state!) |
| **Not Scalable** | Struggles with very large datasets |
| **Only for Visualization** | Results unsuitable for downstream ML (use PCA instead) |
| **Global Structure Lost** | May distort long-distance relationships |
| **Perplexity Sensitive** | Need to tune parameter for each dataset |

## 🎯 When to Use

### Use t-SNE when:
✅ **Exploring data visually** (discovery, understanding clusters)
✅ **Dataset size < 50K samples** (computational feasibility)
✅ **You want beautiful visualizations** for presentations/papers
✅ **Clustering quality matters** more than speed

### Use PCA when:
✅ **Fast dimensionality reduction needed**
✅ **Feeding data into other ML algorithms** (preprocessing)
✅ **Dataset > 100K samples**
✅ **Interpretability matters** (feature importance)

## Applications

Biology: Gene expression analysis, cell type discovery from single-cell RNA-seq data.

NLP: Document embedding visualization, topic clustering, semantic similarity exploration.

Computer Vision: CNN feature visualization, understanding learned representations.

Customer Analytics: Customer segmentation, behavior pattern discovery.

Chemistry: Molecular similarity exploration, compound clustering.

Deep Learning: Neural network layer output visualization, latent space exploration.

## Practical Guidance

For exploratory data analysis with small datasets, t-SNE provides the best visualization. However, it requires careful parameter tuning (especially perplexity) and is computationally expensive.

UMAP offers a modern alternative that scales better while maintaining visualization quality. It is suitable for interactive exploration of datasets up to 100K points.

PCA should be used when the output needs to feed into other machine learning algorithms or when working with very large datasets.

Always standardize data before applying these methods. Use random_state parameter for reproducibility.